# Define Library

In [1]:

# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"
# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)


# Functions

In [2]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

# Count of Onboarded customer as per scorecard 2.0 - New logic Three months from Onboarding

In [3]:
sq = """ 
select 
date(user_timelines.first_account_activated_at) as ee_onboarding_date,
format_date('%Y-%m', date(user_timelines.first_account_activated_at)) Onboarding_month,
t1.employee_id,
t1.oop_risk_segment,
  from prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api t1
  inner join tendopay_raw.user_timelines on cast(user_timelines.user_id as string) = cast(t1.employee_id as string)
   where user_timelines.first_account_activated_at >= '2025-06-01'
        and user_timelines.first_account_activated_at < '2026-01-01'
     qualify row_number() over(partition by t1.employee_id order by t1.run_date) = 1;
"""
df1 = client.query(sq).to_dataframe()

In [4]:
df1.groupby(['Onboarding_month', 'oop_risk_segment']).agg({'employee_id':'nunique'}).reset_index()

,Onboarding_month,oop_risk_segment,employee_id
0,2025-06,A,26
1,2025-06,B,25
2,2025-06,C,27
3,2025-06,D,4
4,2025-06,E,3
5,2025-06,F,7
6,2025-07,A,489
7,2025-07,B,287
8,2025-07,C,137
9,2025-07,D,38


In [5]:

sq = """ 
select user_id, tag_id, tag_name, created_at resignation_date, deleted_at 
from prj-prod-dataplatform.tendopay_raw.user_tag
where tag_id = 100
and deleted_at is null;
"""
userdf = client.query(sq).to_dataframe()


In [6]:
sq = """select user_id, target_maturity_flag, target , ee_resignation_date_correct 
from prj-prod-dataplatform.tendo_mart.tendo_collection_target_master"""
df_target = client.query(sq).to_dataframe()
df_target.shape

(39081, 4)

In [7]:
for col in df1.columns:
    col_type = str(df1[col].dtype).lower()
    
    # Handle date/time types
    if 'dbdate' in col_type or 'date' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col])
        except:
            pass
    elif 'dbtime' in col_type or 'time' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col], format='%H:%M:%S').dt.time
        except:
            pass
    elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col])
        except:
            pass

df2 = dd.query("""select * from df1 
         left join userdf on userdf.user_id = df1.employee_id
            left join df_target on cast(df_target.user_id as string) = cast (df1.employee_id as string)
                   """).to_df()       

In [8]:
df2.head()

,ee_onboarding_date,Onboarding_month,employee_id,oop_risk_segment,user_id,tag_id,tag_name,resignation_date,deleted_at,user_id_1,target_maturity_flag,target,ee_resignation_date_correct
0,2025-07-03,2025-07,1471189,B,1471189,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1471189,0,0,2025-12-29 05:30:00+05:30
1,2025-07-03,2025-07,1470987,A,1470987,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1470987,1,1,2025-12-29 05:30:00+05:30
2,2025-07-05,2025-07,1409198,A,1409198,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1409198,0,0,2025-12-29 05:30:00+05:30
3,2025-06-19,2025-06,1406933,A,1406933,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1406933,1,1,2025-12-29 05:30:00+05:30
4,2025-07-04,2025-07,1398169,A,1398169,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1398169,0,0,2025-12-29 05:30:00+05:30


In [9]:
dd.query("""select * from df2 where date(resignation_date) != date(ee_resignation_date_correct);""").to_df()

,ee_onboarding_date,Onboarding_month,employee_id,oop_risk_segment,user_id,tag_id,tag_name,resignation_date,deleted_at,user_id_1,target_maturity_flag,target,ee_resignation_date_correct
0,2025-07-03,2025-07,1471189,B,1471189,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1471189,0,0,2025-12-29 05:30:00+05:30
1,2025-07-03,2025-07,1470987,A,1470987,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1470987,1,1,2025-12-29 05:30:00+05:30
2,2025-07-05,2025-07,1409198,A,1409198,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1409198,0,0,2025-12-29 05:30:00+05:30
3,2025-06-19,2025-06,1406933,A,1406933,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1406933,1,1,2025-12-29 05:30:00+05:30
4,2025-07-04,2025-07,1398169,A,1398169,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1398169,0,0,2025-12-29 05:30:00+05:30
...,...,...,...,...,...,...,...,...,...,...,...,...,...
282,2025-10-01,2025-10,1568070,C,1568070,100,FREEZE_PERMANENT,2025-12-27 01:00:48+05:30,NaT,1568070,0,0,2025-12-21 05:30:00+05:30
283,2025-06-05,2025-06,1260893,B,1260893,100,FREEZE_PERMANENT,2025-07-29 21:52:44+05:30,NaT,1260893,1,1,2025-07-20 05:30:00+05:30
284,2025-09-12,2025-09,1547519,B,1547519,100,FREEZE_PERMANENT,2025-12-23 00:26:55+05:30,NaT,1547519,0,0,2025-12-22 05:30:00+05:30
285,2025-12-06,2025-12,1644304,B,1644304,100,FREEZE_PERMANENT,2026-01-26 18:42:57+05:30,NaT,1644304,1,1,2026-01-13 05:30:00+05:30


In [10]:
data = dd.query("""select Onboarding_month, oop_risk_segment,
         count(distinct employee_id) as cnt_onboarded_with_sc2,
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) as cnt_left_job_within_Jan_2026,
         count(distinct case when target_maturity_flag = 1 then df2.employee_id end) as cnt_had_oop_outstanding,
         count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end) as cnt_oop_flag_bad_fpd30,
         round(count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end)/
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) *100, 2) as unit_rate_of_all_who_left_job
         from df2
         group by Onboarding_month, oop_risk_segment
         order by Onboarding_month
         """).to_df()

In [11]:
data

,Onboarding_month,oop_risk_segment,cnt_onboarded_with_sc2,cnt_left_job_within_Jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad_fpd30,unit_rate_of_all_who_left_job
0,2025-06,A,26,4,4,1,25.00
1,2025-06,D,4,0,0,0,NaN
2,2025-06,E,3,0,0,0,NaN
3,2025-06,B,25,4,3,1,25.00
4,2025-06,C,27,3,2,1,33.33
5,2025-06,F,7,0,0,0,NaN
6,2025-07,C,137,20,13,9,45.00
7,2025-07,F,140,9,6,3,33.33
8,2025-07,D,38,5,3,3,60.00
9,2025-07,B,287,24,23,13,54.17


In [12]:
data.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Biswa\Tendo_Funnel\20260204_New_Tendo_Funnel_sc2_0_monthlytilldec25.xlsx", sheet_name='New_Tendo_funnel_scorecard2_0')

In [15]:
data2 = dd.query("""select Onboarding_month,
         count(distinct employee_id) as cnt_onboarded_with_sc2,
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) as cnt_left_job_within_Jan_2026,
         count(distinct case when target_maturity_flag = 1 then df2.employee_id end) as cnt_had_oop_outstanding,
         count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end) as cnt_oop_flag_bad_fpd30,
         round(count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end)/
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) *100, 2) as unit_rate_of_all_who_left_job
         from df2
         group by Onboarding_month
         order by 1
         """).to_df()

In [16]:
data2

,Onboarding_month,cnt_onboarded_with_sc2,cnt_left_job_within_Jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad_fpd30,unit_rate_of_all_who_left_job
0,2025-06,92,11,9,3,27.27
1,2025-07,1128,66,81,51,77.27
2,2025-08,863,49,39,23,46.94
3,2025-09,921,55,50,32,58.18
4,2025-10,575,30,14,6,20.00
5,2025-11,951,28,9,0,0.00
6,2025-12,1078,11,2,0,0.00


In [17]:
data2.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Biswa\Tendo_Funnel\20260204_New_Tendo_Funnel_scorecard2_0_monthlydec25.xlsx", sheet_name='New_Tendo_funnel_sc_2_0_rs')

# Peso values of customers Onboarded

## Select the employees who onboarded from June to August 2025

In [18]:
sq = """ 
select distinct
date(user_timelines.first_account_activated_at) as ee_onboarding_date,
format_date('%Y-%m', date(user_timelines.first_account_activated_at)) Onboarding_month,
t1.employee_id,
t1.oop_risk_segment,
  from prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api t1
  inner join tendopay_raw.user_timelines on cast(user_timelines.user_id as string) = cast(t1.employee_id as string)
   where user_timelines.first_account_activated_at >= '2025-06-01'
     and user_timelines.first_account_activated_at < '2026-01-01'
  qualify row_number() over(partition by t1.employee_id order by t1.run_date) = 1;
"""
df1 = client.query(sq).to_dataframe()
print(f"The data for the customers who onboarded between June to August 2025 is: {df1.shape}")

The data for the customers who onboarded between June to August 2025 is: (5608, 4)


In [19]:
df1.groupby('Onboarding_month').agg({'employee_id':'nunique'}).reset_index()

,Onboarding_month,employee_id
0,2025-06,92
1,2025-07,1128
2,2025-08,863
3,2025-09,921
4,2025-10,575
5,2025-11,951
6,2025-12,1078


In [20]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

In [21]:
convert_datetime_columns(df1)

In [22]:
dd.query("""select employee_id, count(employee_id) as emp_count from df1 group by employee_id having emp_count > 1 order by 2 desc;""").to_df()

,employee_id,emp_count


In [23]:
sq = """ 
with frozen_tags AS (
  SELECT
    user_id,
    max(CASE WHEN tag_id IN (39, 100, 101, 102, 103) THEN 1 ELSE 0 END)
      AS frozen_tag
  FROM `tendopay_raw.user_tag`
  GROUP BY user_id
),
employee_info as
(SELECT
    u.id user_id,
    u.created_at sign_up_date,
    ut.first_account_activated_at AS ee_onboarding_date,
    ut.first_account_activated_at approval_date,
    u.employer_id employer_id,
    e.name employer_name,
    ii.employment_date employment_date,
    LENGTH(employment_date) LENGTH_employment_date,
    datetime_diff(
      CAST(ii.created_at AS date),
      CASE  ------original: datetime_diff(cast(ut.first_account_activated_at as date),
        WHEN LENGTH(employment_date) = 6
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 1) AS int64),
              1)
        WHEN LENGTH(employment_date) = 7
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 2) AS int64),
              1)
        WHEN LENGTH(employment_date) = 10
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 2) AS int64),
              CAST(substr(employment_date, 9, 2) AS int64))
        END,
      day)
      / 365
      AS employer_time,  ----sometimes employment_date is missing, sometimes approval_date (first_account_activated_at) is missing
    u.gender gender,
    u.civil_status civil_status,
    ii.employee_status employment_status,
    CASE
      WHEN ii.income IN ('0-10000') THEN 5000
      WHEN ii.income IN ('10000-20000') THEN 15000
      WHEN ii.income IN ('20000-30000') THEN 25000
      WHEN ii.income IN ('30000-40000') THEN 35000
      WHEN ii.income IN ('40000-50000') THEN 45000
      WHEN ii.income IN ('50000+') THEN 50000
      ELSE CAST(ii.income AS numeric)
      END AS declared_income_num,
    ii.verified_net_income verified_net_income,
    CASE
      WHEN tag.frozen_tag = 1 THEN 'Frozen'
      ELSE 'Not Frozen'
      END Frozen_Status,

      cci.value                                                                               Credit_limit,
       
  FROM `tendopay_raw.users` u
  LEFT JOIN `tendopay_raw.income_info` ii
    ON u.id = ii.user_id
  LEFT JOIN `tendopay_raw.user_timelines` ut
    ON u.id = ut.user_id
  LEFT JOIN `tendopay_raw.employers` e
    ON u.employer_id = CAST(e.id AS string)
  LEFT JOIN frozen_tags tag
    ON u.id = tag.user_id

  LEFT JOIN `tendopay_raw.customer_credit_information` cci on u.id = cci.user_id
  WHERE u.product_type IN ('employer', 'payroll')

  and u.account_activated = 2
  and cci.`key` = 'credit-limit'
)
select user_id, 
sign_up_date,
ee_onboarding_date,
employer_id,
employment_date,
LENGTH_employment_date,
gender,
civil_status,
employment_status,
declared_income_num,
verified_net_income,
Frozen_Status,
Credit_limit,
 from employee_info
;
"""

df_employee = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The data for the employee info is: {df_employee.shape}")


Job ID 136d223d-2e37-4070-9088-f44e7be67e3d successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The data for the employee info is: (185751, 13)


In [24]:
df_employee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185751 entries, 0 to 185750
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   user_id                 185751 non-null  Int64              
 1   sign_up_date            185751 non-null  datetime64[us, UTC]
 2   ee_onboarding_date      185749 non-null  datetime64[us, UTC]
 3   employer_id             185680 non-null  object             
 4   employment_date         185730 non-null  object             
 5   LENGTH_employment_date  185730 non-null  Int64              
 6   gender                  185716 non-null  object             
 7   civil_status            184498 non-null  object             
 8   employment_status       174661 non-null  object             
 9   declared_income_num     185734 non-null  object             
 10  verified_net_income     185595 non-null  object             
 11  Frozen_Status           18

In [25]:
df1.columns

Index(['ee_onboarding_date', 'Onboarding_month', 'employee_id',
       'oop_risk_segment'],
      dtype='object')

In [26]:
df1['user_id'] = df1['employee_id'].astype('int64')

## Merging Onboarded employee from June 2025 to August 2025 with employee info data

In [27]:
data = df1.merge(df_employee, left_on='user_id', right_on='user_id', how='left')
print(f"The shape of the data after merging employee info is: {data.shape}")

The shape of the data after merging employee info is: (5608, 17)


In [28]:
data.head()

,ee_onboarding_date_x,Onboarding_month,employee_id,oop_risk_segment,user_id,sign_up_date,ee_onboarding_date_y,employer_id,employment_date,LENGTH_employment_date,gender,civil_status,employment_status,declared_income_num,verified_net_income,Frozen_Status,Credit_limit
0,2025-07-09,2025-07,1188966,B,1188966,2024-04-27 15:23:26+00:00,2025-07-09 13:45:32+00:00,10161,2024-01,7,Male,Single,regular,18000.000000000,9000,Not Frozen,25200.00
1,2025-07-23,2025-07,1327703,B,1327703,2025-01-10 08:49:10+00:00,2025-07-23 14:59:22+00:00,10239,2024-01,7,Male,Single,regular,15000.000000000,18000,Not Frozen,17608.00
2,2025-12-16,2025-12,1330525,B,1330525,2025-01-16 20:36:34+00:00,2025-12-16 14:57:53+00:00,10048,2024-07,7,Male,Single,regular,38000.000000000,16000,Frozen,43344
3,2025-12-17,2025-12,1375921,B,1375921,2025-04-02 00:43:11+00:00,2025-12-17 14:42:03+00:00,10048,2024-07,7,Male,Single,regular,33000.000000000,17500,Frozen,39747.00
4,2025-07-04,2025-07,1471269,A,1471269,2025-07-01 14:26:32+00:00,2025-07-04 11:40:41+00:00,10224,2024-01,7,Male,Single,regular,25000.000000000,19000,Not Frozen,18791.00


In [29]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

check duplicate

In [30]:
convert_datetime_columns(data)
dd.query("""select employee_id, count(employee_id) as emp_count from data group by employee_id having emp_count > 1 order by 2 desc;""").to_df()

,employee_id,emp_count


No duplicates found

## OOP maturity target merged with Oleh's backscore table

In [31]:
sq = """ 
with b1 as 
(select user_id, target_maturity_flag, target , ee_resignation_date_correct 
from prj-prod-dataplatform.tendo_mart.tendo_collection_target_master
where target_maturity_flag = 1
)
select * from b1 
left join risk_mart.tendo_backscored_new_users_jan23_jan26_20260201_oop_with_osbal b2 on cast(b1.user_id as numeric) = cast(b2.ee_customer_id as numeric);
"""
df_target = client.query(sq).to_dataframe()
print(f"The data for the target maturity info is: {df_target.shape}")

The data for the target maturity info is: (31272, 11)


In [32]:
df_target['user_id'] = df_target['user_id'].astype('int64')

Check for duplicates

In [33]:
dd.query("""select user_id, count(user_id) cnt from df_target group by 1 having count(user_id) > 1 order by 2 desc;""").to_df()

,user_id,cnt


No Duplicates found

No duplicate found in data dataframe

In [34]:
data1 = data.merge(df_target, left_on='user_id', right_on='user_id', how='inner')

In [35]:
data1.head()

,ee_onboarding_date_x,Onboarding_month,employee_id,oop_risk_segment,user_id,sign_up_date,ee_onboarding_date_y,employer_id,employment_date,LENGTH_employment_date,gender,civil_status,employment_status,declared_income_num,verified_net_income,Frozen_Status,Credit_limit,target_maturity_flag,target,ee_resignation_date_correct,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
0,2025-09-05,2025-09,1504229,A,1504229,2025-08-01 20:25:47+00:00,2025-09-05 17:53:26+00:00,10224,2025-05,7,Female,Single,probationary,22000.000000000,19500,Frozen,12159.00,1,0,2025-09-30 00:00:00+00:00,<NA>,NaT,NaN,NaN,NaN,NaN,NaN
1,2025-07-03,2025-07,1471483,A,1471483,2025-07-01 18:20:39+00:00,2025-07-03 11:40:58+00:00,10019,2023-10,7,Male,Single,regular,32500.000000000,22000,Frozen,55000.00,1,0,2025-09-22 00:00:00+00:00,1471483,2025-08-01,0.0,0.558906,16794.767,16794.767,8555.322
2,2025-09-01,2025-09,1462062,E,1462062,2025-06-20 09:57:53+00:00,2025-09-01 12:39:53+00:00,10200,2024-07,7,Female,Single,regular,35000.000000000,18000,Frozen,14086.00,1,1,2025-11-15 00:00:00+00:00,1462062,2025-10-01,NaN,0.413524,10710.087,10710.087,11795.289
3,2025-10-21,2025-10,1568352,D,1568352,2025-10-01 20:24:58+00:00,2025-10-21 10:52:18+00:00,10065,2024-08,7,Male,Single,regular,24000.000000000,19500,Frozen,10158,1,0,2025-11-10 00:00:00+00:00,1568352,2025-11-01,NaN,0.439990,0.000,7728.404,7728.404
4,2025-07-22,2025-07,1492842,A,1492842,2025-07-22 11:08:31+00:00,2025-07-22 12:03:32+00:00,10107,2025-02,7,Female,Married,regular,80000.000000000,78500,Frozen,31400,1,1,2025-09-03 00:00:00+00:00,1492842,2025-08-01,1.0,0.642407,20373.755,3747.800,0.000


In [36]:
dd.query("""select employee_id, count(employee_id) cnt from data1 group by 1 having count(employee_id) > 1 order by 2 desc;""").to_df()

,employee_id,cnt


In [37]:
data3 = dd.query(f""" select Onboarding_month, oop_risk_segment,
         -- count(distinct employee_id) as cnt_onboarded_with_sc2,
         sum(cast(Credit_limit as numeric)) Credit_limit,
         -- count(distinct case when ee_resignation_date_correct is not null then employee_id end) as cnt_resigned_employees,
         -- count(distinct case when target_maturity_flag = 1 then employee_id end) as cnt_had_oop_eligible_cnt,
         -- count(distinct case when target = 0 then employee_id end) as cnt_oop_flag_bad_fpd30,
         -- count(distinct case when target = 1 then employee_id end) as cnt_oop_flag_good_fpd30,
         sum(cast(osbal_as_of_resignation_date as numeric)) as sum_oop_outstanding_resignation_date,
         sum(case when target_maturity_flag = 1 then cast(osbal_as_of_oop_eligible_date as numeric) else 0 end) as osbal_as_of_oop_eligible_date,
         sum(case when target = 0 then cast(coalesce(osbal_as_of_current_date, 0) as numeric) end) as tot_os_principal_amt_from_bad_customers,
         from data1
            group by Onboarding_month, oop_risk_segment
                 order by Onboarding_month, oop_risk_segment
         """
         ).to_df()

In [38]:
data3

,Onboarding_month,oop_risk_segment,Credit_limit,sum_oop_outstanding_resignation_date,osbal_as_of_oop_eligible_date,tot_os_principal_amt_from_bad_customers
0,2025-06,A,99475.0,35589.669,19625.777,4509.748
1,2025-06,B,106970.0,80082.339,76500.862,17378.052
2,2025-06,C,51595.0,41756.014,41756.014,26157.168
3,2025-07,A,919594.5,317693.875,246357.207,167276.802
4,2025-07,B,579545.0,156685.080,187271.452,136908.299
5,2025-07,C,196563.0,78969.836,83750.057,62766.933
6,2025-07,D,60734.0,14588.733,NaN,0.000
7,2025-07,E,135127.0,12058.406,19275.008,9968.586
8,2025-07,F,263558.0,33898.732,35917.573,33387.967
9,2025-08,A,334500.0,67345.901,59062.228,47979.873


In [39]:
data3.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Biswa\Tendo_Funnel\20260204_New_Tendo_Funnel_scorecard2_0_peso_monthly_dec25.xlsx", sheet_name='New_Tendo_funnel_sc2_0_peso')

In [40]:
data2 = dd.query(f"""select Onboarding_month,
         sum(cast(Credit_limit as numeric)) tot_cl_amt_onboarded_with_2_0,
         --- count(distinct case when ee_resignation_date_correct is not null then employee_id end) as cnt_resigned_employees,
         --- count(distinct case when target_maturity_flag = 1 then employee_id end) as cnt_had_oop_eligible_cnt,
         --- count(distinct case when target = 0 then employee_id end) as cnt_oop_flag_bad_fpd30,
         --- count(distinct case when target = 1 then employee_id end) as cnt_oop_flag_good_fpd30,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-02-01' then cast(osbal_as_of_resignation_date as numeric) else 0 end) as tot_os_principal_amt_as_of_resignation_date,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-02-01' and target_maturity_flag = 1 then cast(osbal_as_of_oop_eligible_date as numeric) else 0 end) as tot_os_principal_amt_as_of_oop_eligible_dt,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-02-01' and target_maturity_flag = 1 and target = 0 then cast(coalesce(osbal_as_of_current_date, 0) as numeric) end) as tot_os_principal_amt_from_bad_customers,
         from data1
                 group by 1
                 order by 1
                   """
         ).to_df()

In [42]:
data2.sort_values(by='Onboarding_month')

,Onboarding_month,tot_cl_amt_onboarded_with_2_0,tot_os_principal_amt_as_of_resignation_date,tot_os_principal_amt_as_of_oop_eligible_dt,tot_os_principal_amt_from_bad_customers
0,2025-06,258040.0,157428.022,137882.653,48044.968
1,2025-07,2155121.5,613894.662,572571.297,410308.587
2,2025-08,829277.0,244732.539,236988.545,173890.278
3,2025-09,1088727.0,409056.498,391863.705,273221.864
4,2025-10,311806.0,65428.946,65758.360,39842.054
5,2025-11,212136.0,23898.559,14525.934,NaN
6,2025-12,24643.0,NaN,NaN,NaN


In [43]:
data2.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Biswa\Tendo_Funnel\20260204_New_Tendo_Funnel_scorecard2_0_peso_dec25.xlsx", sheet_name='New_Tendo_funnel_sc2_0_peso')